In [1]:
import os

import pandas as pd
import requests
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", None)


# Extract

In [2]:
def extract_table(
    url: str = "https://www.vlr.gg/event/stats/2766/valorant-champions-2026",
    tag: str = "table",
    class_tag: str = "st-table mod-hide-maps mod-hide-fbpr mod-hide-fdpr mod-hide-fk mod-hide-fd",
):
    try:
        response = requests.get(url, timeout=5)
        # Explicitly raise an error for 4xx or 5xx status codes
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        # Catches ConnectionError, Timeout, HTTPError, etc.
        print(f"A requests error occurred: {e}")

    soup = BeautifulSoup(response.text, "lxml")
    table = soup.find(
        tag,
        class_=class_tag,
    )

    headers = [header.text.strip().lower() for header in table.find_all("th")]
    headers.append("agents_list")

    df = pd.DataFrame(columns=headers)

    for row in table.find_all("tr")[1:]:
        data = row.find_all("td")
        row_data = [td.text.strip() for td in data]

        # Skip empty rows (like header rows inside tbody)
        if not row_data:
            continue

        agents_list = row.find_all("img")
        agents = [img["src"].split("/")[-1].split(".")[0] for img in agents_list]
        agents_str = ", ".join(agents)

        # Combine text data and agents into a single row
        full_row = row_data + [agents_str]

        # Append to DataFrame
        df.loc[len(df)] = full_row

    return df


In [3]:
df = extract_table()


In [6]:
df.info()


<class 'pandas.DataFrame'>
Index: 80 entries, 0 to 79
Data columns (total 24 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   player       80 non-null     str  
 1   agents       80 non-null     str  
 2   maps         80 non-null     str  
 3   rnd          80 non-null     str  
 4   r            80 non-null     str  
 5   acs          80 non-null     str  
 6   k:d          80 non-null     str  
 7   kast         80 non-null     str  
 8   adr          80 non-null     str  
 9   kpr          80 non-null     str  
 10  apr          80 non-null     str  
 11  fk:fd        80 non-null     str  
 12  fk%          80 non-null     str  
 13  fd%          80 non-null     str  
 14  hs%          80 non-null     str  
 15  cl%          80 non-null     str  
 16  cl           80 non-null     str  
 17  kmax         80 non-null     str  
 18  k            80 non-null     str  
 19  d            80 non-null     str  
 20  a            80 non-null    

In [ ]:
def load_to_parquet(
    df: pd.DataFrame,
    file_path: str,
    compression: str = "gzip",
    keep_index: bool = False,
) -> None:
    """
    Saves a pandas DataFrame to a Parquet file safely.

    Parameters:
        df (pd.DataFrame): The DataFrame to save.
        file_path (str): Target path for the .parquet file.
        compression (str): Compression type ('snappy', 'gzip', 'brotli', 'lz4', 'zstd', or None).
        keep_index (bool): Whether to include the DataFrame's index in the Parquet file.
    """

    try:
        # Ensure the directory exists
        directory = os.path.dirname(file_path)
        if directory and not os.path.exists(directory):
            os.makedirs(directory)

        # Write DataFrame to Parquet
        df.to_parquet(
            path=file_path, engine="pyarrow", compression=compression, index=keep_index
        )
        print(f"Successfully saved DataFrame to {file_path}")

    except FileExistsError as e:
        print(f"Error saving DataFrame to Parquet: {e}")


In [ ]:
load_to_parquet(df, r"data\raw\raw_data.parquet", compression="gzip", keep_index=False)


Successfully saved DataFrame to data\raw\raw_data.parquet


# Transform

In [ ]:
def transform_data(df: pd.DataFrame) -> pd.DataFrame:

    if df.empty:
        print("Input DataFrame is empty. Returning empty DataFrame.")

    clean_df = df.copy()

    clean_df = clean_df.drop_duplicates()
    # Get player name and its team by splitting "player" column
    clean_df[["player_name", "team"]] = clean_df["player"].str.split("\n", expand=True)

    rename_columns = {
        # "agents_list": "",
        # "agents": "",
        "maps": "maps_played",
        "rnd": "rounds_played",
        "r": "rating",
        "acs": "avg_combat_score",
        "k:d": "kills_deaths",
        # "kast": "",
        "adr": "avg_damage_per_round",
        "kpr": "kills_per_round",
        "apr": "assists_per_round",
        "fk:fd": "first_kills_first_deaths",
        "fkpr": "first_kills_per_round",
        "fdpr": "firsts_deaths_per_round",
        "hs%": "headshot_pct",
        "cl%": "clutch_pct",
        "cl": "clucth_won",
        "kmax": "most_kilss_in_map",
        "k": "total_kills",
        "d": "total_deaths",
        "a": "total_assists",
        "fk": "total_first_kills",
        "fd": "total_first_deaths",
    }
    df.rename(columns=rename_columns)

    filter_columns = [
        "player_name",
        "team",
        "agents_list",
        # "player",
        # "agents",
        "maps_played",
        "rounds_played",
        "rating",
        "avg_combat_score",
        "kills_deaths",
        "kast",
        "avg_damage_per_round",
        "kills_per_round",
        "assists_per_round",
        "first_kills_first_deaths",
        "first_kills_per_round",
        "firsts_deaths_per_round",
        "headshot_pct",
        "clutch_pct",
        "clucth_won",
        "most_kilss_in_map",
        "total_kills",
        "total_deaths",
        "total_assists",
        "total_first_kills",
        "total_first_deaths",
    ]

    df = df[filter_columns]


In [12]:
def to_numeric_safe(series, strip_percent=False):
    s = series.astype(str).str.strip()
    if strip_percent:
        s = s.str.replace("%", "", regex=False)

    s = s.replace(
        {
            "—": pd.NA,
            "–": pd.NA,
            "-": pd.NA,
            "--": pd.NA,
            "N/A": pd.NA,
            "n/a": pd.NA,
            "NaN": pd.NA,
        }
    )
    s = s.str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")


df["player_name"] = df["player_name"].fillna("").astype(str).str.strip()
df["team"] = df["team"].fillna("").astype(str).str.strip()
df["agents_list"] = df["agents_list"].fillna("").astype(str).str.strip()
df["maps_played"] = to_numeric_safe(df["maps_played"]).astype("Int64")
df["rounds_played"] = to_numeric_safe(df["rounds_played"]).astype("Int64")
df["rating"] = to_numeric_safe(df["rating"])
df["avg_combat_score"] = to_numeric_safe(df["avg_combat_score"]).astype("Int64")
df["kills_deaths"] = to_numeric_safe(df["kills_deaths"])
df["kast"] = to_numeric_safe(df["kast"], strip_percent=True).astype("Int64")
df["avg_damage_per_round"] = to_numeric_safe(df["avg_damage_per_round"])
df["kills_per_round"] = to_numeric_safe(df["kills_per_round"])
df["assists_per_round"] = to_numeric_safe(df["assists_per_round"])
df["first_kills_first_deaths"] = to_numeric_safe(df["first_kills_first_deaths"])
df["first_kills_per_round"] = to_numeric_safe(df["first_kills_per_round"])
df["firsts_deaths_per_round"] = to_numeric_safe(df["firsts_deaths_per_round"])
df["headshot_pct"] = to_numeric_safe(df["headshot_pct"], strip_percent=True)
df["clutch_pct"] = to_numeric_safe(df["clutch_pct"], strip_percent=True)
df["clucth_won"] = df["clucth_won"].fillna("").astype(str)
df["most_kilss_in_map"] = to_numeric_safe(df["most_kilss_in_map"]).astype("Int64")
df["total_kills"] = to_numeric_safe(df["total_kills"]).astype("Int64")
df["total_deaths"] = to_numeric_safe(df["total_deaths"]).astype("Int64")
df["total_assists"] = to_numeric_safe(df["total_assists"]).astype("Int64")
df["total_first_kills"] = to_numeric_safe(df["total_first_kills"]).astype("Int64")
df["total_first_deaths"] = to_numeric_safe(df["total_first_deaths"]).astype("Int64")


In [13]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
maps_played,80.0,2.25,0.435745,2.0,2.0,2.0,2.25,3.0
rounds_played,80.0,44.875,11.911461,35.0,36.5,40.0,47.25,69.0
rating,80.0,0.989,0.272757,0.29,0.83,0.99,1.155,1.66
avg_combat_score,80.0,192.825,45.266517,86.0,159.5,189.5,217.25,321.0
kills_deaths,80.0,1.057875,0.4204,0.33,0.7875,0.98,1.2925,2.24
kast,80.0,69.75,10.833025,46.0,62.75,70.0,78.0,91.0
avg_damage_per_round,80.0,125.98375,27.579016,51.9,103.6,128.35,143.425,208.2
kills_per_round,80.0,0.678375,0.179291,0.28,0.57,0.665,0.77,1.21
assists_per_round,80.0,0.2625,0.133838,0.03,0.16,0.24,0.3325,0.62
first_kills_first_deaths,77.0,1.236494,1.153524,0.0,0.55,1.0,1.5,7.0


# Load

In [ ]:
load_to_parquet(
    df, r"data\silver\silver_data.parquet", compression="gzip", keep_index=False
)


Successfully saved DataFrame to data\silver\silver_data.parquet
